# Candle Prediction using Market Depth

In [94]:
from pathlib import Path
import pandas as pd
import numpy as np
from dev.utils import resample_fractional_minute, apply_trailing_logic, generate_signal

In [95]:
# ---- Input ------
date_ = "23APR2026"
file_name = "NIFTY26APR24200CE.xlsx"

file_path = Path(fr"D:\Study\Programs\trading\assets\logs\{date_}\extracted_symbols\{file_name}")
df = pd.read_excel(file_path)
if "PE" in file_name or "CE" in file_name:
    col_name = "last_trade_time"

    # Volume computation
    # 1. Calculate the basic difference between rows
    df["volume_at_tick"] = df["volume_traded"].diff()
    df.loc[df["volume_at_tick"] == 0, "volume_at_tick"] = np.nan
    df["volume_at_tick"] = df["volume_at_tick"].ffill()
    df["volume_at_tick"] = df["volume_at_tick"].fillna(0)
else:
    col_name = "local_time"

df[col_name] = pd.to_datetime(df[col_name])
target_date = pd.to_datetime(date_).date()
df = df[df[col_name].dt.date == target_date]

In [96]:
file_name

'NIFTY26APR24200CE.xlsx'

In [97]:
df.head(4)

,instrument_token,symbol,exchange_timestamp,local_time,last_trade_time,last_price,last_traded_quantity,average_traded_price,option_CE_PE,option_type,...,total_sell_quantity,ohlc,change,oi,oi_day_high,oi_day_low,depth,tradable,mode,volume_at_tick
2,18501378,NIFTY26APR24200CE,NaN,2026-04-23 09:15:00.619,2026-04-23 09:15:00,194.95,65,193.46,CE,atm,...,16315,"{'open': 237.0, 'high': 237.0, 'low': 186.45, ...",-40.491453,1252485,1252485,1252485,"{'buy': [{'quantity': 520, 'price': 186.95, 'o...",True,full,15665.0
3,18501378,NIFTY26APR24200CE,NaN,2026-04-23 09:15:00.619,2026-04-23 09:15:00,194.95,65,193.46,CE,atm,...,16315,"{'open': 237.0, 'high': 237.0, 'low': 186.45, ...",-40.491453,1252485,1252485,1252485,"{'buy': [{'quantity': 520, 'price': 186.95, 'o...",True,full,15665.0
4,18501378,NIFTY26APR24200CE,NaN,2026-04-23 09:15:01.369,2026-04-23 09:15:00,190.10,65,193.46,CE,atm,...,16315,"{'open': 237.0, 'high': 237.0, 'low': 186.45, ...",-41.971917,1252485,1252485,1252485,"{'buy': [{'quantity': 520, 'price': 186.95, 'o...",True,full,15665.0
5,18501378,NIFTY26APR24200CE,NaN,2026-04-23 09:15:01.369,2026-04-23 09:15:00,190.10,65,193.46,CE,atm,...,16315,"{'open': 237.0, 'high': 237.0, 'low': 186.45, ...",-41.971917,1252485,1252485,1252485,"{'buy': [{'quantity': 520, 'price': 186.95, 'o...",True,full,15665.0


In [98]:
# ---- Input ------
N = 6
window = 3
price_pct_threshold = 0.001
volume_threshold=1000
volume_period = 30

use_price_pct_level=True
use_price_trend=True
use_volume=True

In [99]:
df.columns

Index(['instrument_token', 'symbol', 'exchange_timestamp', 'local_time',
       'last_trade_time', 'last_price', 'last_traded_quantity',
       'average_traded_price', 'option_CE_PE', 'option_type', 'strike',
       'volume_traded', 'total_buy_quantity', 'total_sell_quantity', 'ohlc',
       'change', 'oi', 'oi_day_high', 'oi_day_low', 'depth', 'tradable',
       'mode', 'volume_at_tick'],
      dtype='str')

In [100]:
print(type(df.iloc[0]["local_time"]))
print(df.iloc[0]["local_time"])
print(df.iloc[0]["last_trade_time"])

<class 'pandas.Timestamp'>
2026-04-23 09:15:00.619000
2026-04-23 09:15:00


In [101]:
clubbed_df = resample_fractional_minute(df, col_name, N)
clubbed_df["bucket_time_next"] = clubbed_df["bucket_time"].shift(-1)
clubbed_df["price_diff"] = clubbed_df["close"] - clubbed_df["open"]
clubbed_df["price_pct"] = (clubbed_df["close"] - clubbed_df["open"])/clubbed_df["open"]

clubbed_df["volume_diff"] = clubbed_df["volume_high"] - clubbed_df["volume_low"]
clubbed_df["volume_participated"] = clubbed_df["volume_high"] - clubbed_df["volume_low"]
clubbed_df["volume_pct"] = (clubbed_df["volume_close"] - clubbed_df["volume_open"])/clubbed_df["volume_open"]
clubbed_df["volume_avg"] = clubbed_df["volume_close"].rolling(volume_period).median()

In [102]:
clubbed_df[clubbed_df['bucket_time'].dt.minute == 0].head()

,bucket_time,minute,open,high,low,close,volume_open,volume_high,volume_low,volume_close,...,minute_high,minute_low,minute_close,bucket_time_next,price_diff,price_pct,volume_diff,volume_participated,volume_pct,volume_avg
270,2026-04-23 10:00:00,2026-04-23 10:00:00,243.20,243.65,241.00,243.65,10010.0,10010.0,2015.0,8840.0,...,243.65,235.65,235.8,2026-04-23 10:00:10,0.45,0.001850,7995.0,7995.0,-0.116883,5817.5
271,2026-04-23 10:00:10,2026-04-23 10:00:00,242.30,243.60,241.25,241.25,5070.0,5525.0,1820.0,2275.0,...,243.65,235.65,235.8,2026-04-23 10:00:20,-1.05,-0.004333,3705.0,3705.0,-0.551282,5590.0
272,2026-04-23 10:00:20,2026-04-23 10:00:00,241.95,242.90,241.10,241.10,3965.0,6630.0,1430.0,6630.0,...,243.65,235.65,235.8,2026-04-23 10:00:30,-0.85,-0.003513,5200.0,5200.0,0.672131,5590.0
273,2026-04-23 10:00:30,2026-04-23 10:00:00,241.55,241.80,238.60,240.20,1690.0,9620.0,1235.0,4680.0,...,243.65,235.65,235.8,2026-04-23 10:00:40,-1.35,-0.005589,8385.0,8385.0,1.769231,5102.5
274,2026-04-23 10:00:40,2026-04-23 10:00:00,239.65,239.65,236.10,236.10,2665.0,39195.0,2275.0,6305.0,...,243.65,235.65,235.8,2026-04-23 10:00:50,-3.55,-0.014813,36920.0,36920.0,1.365854,5102.5


In [103]:
clubbed_df[["volume_open", "volume_high", "volume_low", "volume_close", "volume_participated"]]

,volume_open,volume_high,volume_low,volume_close,volume_participated
0,15665.0,118235.0,15665.0,74360.0,102570.0
1,68380.0,81900.0,40625.0,40625.0,41275.0
2,46735.0,46735.0,30810.0,30810.0,15925.0
3,40690.0,89635.0,20995.0,33410.0,68640.0
4,51935.0,53885.0,25935.0,26260.0,27950.0
...,...,...,...,...,...
2245,24505.0,27755.0,12285.0,12285.0,15470.0
2246,19565.0,24960.0,15405.0,18980.0,9555.0
2247,25870.0,25870.0,7800.0,14690.0,18070.0
2248,17485.0,17485.0,7215.0,11115.0,10270.0


In [104]:
clubbed_df[clubbed_df['bucket_time'].dt.minute == 0].shape

(36, 22)

In [105]:
# generate_signal
clubbed_df_2 = generate_signal(clubbed_df, window, price_pct_threshold = price_pct_threshold, volume_threshold=volume_threshold,
                               use_price_pct_level=True, use_price_trend=True, use_volume=False)

use_price_pct_level :  True
use_price_trend :  True
use_volume :  False


In [106]:
clubbed_df_2[clubbed_df_2["predicted"]==clubbed_df["candle_type"]].tail(5)

,bucket_time,minute,open,high,low,close,volume_open,volume_high,volume_low,volume_close,...,minute_low,minute_close,bucket_time_next,price_diff,price_pct,volume_diff,volume_participated,volume_pct,volume_avg,predicted
2194,2026-04-23 15:20:40,2026-04-23 15:20:00,168.65,169.80,168.05,169.35,4875.0,18655.0,4875.0,6825.0,...,166.00,170.35,2026-04-23 15:20:50,0.70,0.004151,13780.0,13780.0,0.400000,8905.0,BUY
2195,2026-04-23 15:20:50,2026-04-23 15:20:00,169.75,170.60,169.70,170.35,9490.0,34580.0,9490.0,10335.0,...,166.00,170.35,2026-04-23 15:21:00,0.60,0.003535,25090.0,25090.0,0.089041,8905.0,BUY
2203,2026-04-23 15:22:10,2026-04-23 15:22:00,169.05,169.90,168.85,168.85,6565.0,6565.0,1885.0,1885.0,...,167.40,167.75,2026-04-23 15:22:20,-0.20,-0.001183,4680.0,4680.0,-0.712871,6825.0,SELL
2227,2026-04-23 15:26:10,2026-04-23 15:26:00,168.40,168.70,167.60,167.80,10010.0,26455.0,7215.0,22295.0,...,167.00,169.20,2026-04-23 15:26:20,-0.60,-0.003563,19240.0,19240.0,1.227273,6012.5,SELL
2249,2026-04-23 15:29:50,2026-04-23 15:29:00,167.00,167.95,165.45,165.95,29575.0,29575.0,3315.0,3315.0,...,165.45,165.95,NaT,-1.05,-0.006287,26260.0,26260.0,-0.887912,7247.5,SELL


In [107]:
# df_with_signal = df.merge(
#         clubbed_df[["bucket_time", "predicted"]],
#         left_on="last_trade_time",
#         right_on="bucket_time",
#         how="left"
#     )

# import pandas as pd

# 1. Ensure both DataFrames are sorted by the time columns
df = df.sort_values("last_trade_time")
clubbed_df_2 = clubbed_df_2.sort_values("bucket_time")

# 2. Perform the proximity merge
df_with_signal = pd.merge_asof(
    df,
    clubbed_df_2[["bucket_time", "predicted"]],
    left_on="last_trade_time",
    right_on="bucket_time",
    direction="backward" # Only looks at the past/current, never the future
)

# Find duplicates in bucket_time and set their 'predicted' value to NaN
df_with_signal.loc[df_with_signal.duplicated(subset=['bucket_time'], keep='first'), 'predicted'] = np.nan

In [108]:
# df_with_signal.to_excel("df_with_signal.xlsx")

In [109]:
# clubbed_df_2.to_excel("clubbed_df_2.xlsx")

In [110]:
df_with_signal.head()

,instrument_token,symbol,exchange_timestamp,local_time,last_trade_time,last_price,last_traded_quantity,average_traded_price,option_CE_PE,option_type,...,change,oi,oi_day_high,oi_day_low,depth,tradable,mode,volume_at_tick,bucket_time,predicted
0,18501378,NIFTY26APR24200CE,NaN,2026-04-23 09:15:00.619,2026-04-23 09:15:00,194.95,65,193.46,CE,atm,...,-40.491453,1252485,1252485,1252485,"{'buy': [{'quantity': 520, 'price': 186.95, 'o...",True,full,15665.0,2026-04-23 09:15:00,NaN
1,18501378,NIFTY26APR24200CE,NaN,2026-04-23 09:15:00.619,2026-04-23 09:15:00,194.95,65,193.46,CE,atm,...,-40.491453,1252485,1252485,1252485,"{'buy': [{'quantity': 520, 'price': 186.95, 'o...",True,full,15665.0,2026-04-23 09:15:00,NaN
2,18501378,NIFTY26APR24200CE,NaN,2026-04-23 09:15:01.369,2026-04-23 09:15:00,190.10,65,193.46,CE,atm,...,-41.971917,1252485,1252485,1252485,"{'buy': [{'quantity': 520, 'price': 186.95, 'o...",True,full,15665.0,2026-04-23 09:15:00,NaN
3,18501378,NIFTY26APR24200CE,NaN,2026-04-23 09:15:01.369,2026-04-23 09:15:00,190.10,65,193.46,CE,atm,...,-41.971917,1252485,1252485,1252485,"{'buy': [{'quantity': 520, 'price': 186.95, 'o...",True,full,15665.0,2026-04-23 09:15:00,NaN
4,18501378,NIFTY26APR24200CE,NaN,2026-04-23 09:15:02.120,2026-04-23 09:15:01,184.05,130,191.93,CE,atm,...,-43.818681,1252485,1252485,1252485,"{'buy': [{'quantity': 910, 'price': 182.55, 'o...",True,full,105430.0,2026-04-23 09:15:00,NaN


In [111]:
df_with_signal.columns

Index(['instrument_token', 'symbol', 'exchange_timestamp', 'local_time',
       'last_trade_time', 'last_price', 'last_traded_quantity',
       'average_traded_price', 'option_CE_PE', 'option_type', 'strike',
       'volume_traded', 'total_buy_quantity', 'total_sell_quantity', 'ohlc',
       'change', 'oi', 'oi_day_high', 'oi_day_low', 'depth', 'tradable',
       'mode', 'volume_at_tick', 'bucket_time', 'predicted'],
      dtype='str')

In [112]:
params = {
    "initial_sl_pct": 0.02,
    "target_pct": 0.01,
    "trail_sl_pct": 0.02,
    "tight_sl_offset": 0.5,
}

In [113]:
trades = apply_trailing_logic(df_with_signal, params)

trades = pd.DataFrame(trades)
if len(trades):
    trades["final"] = trades.apply(lambda row: "profit" if row["profit"] > 0 else "loss", axis=1)
else:
    print("trades not generated")

In [114]:
len(trades)

81

In [115]:
# trades

In [116]:
trades[trades["final"]=="profit"]["profit"].sum()

np.float64(171.80000000000007)

In [117]:
trades[trades["final"]=="loss"]["profit"].sum()

np.float64(-22.726)

In [118]:
trades.head(11)

,entry_time,entry_price,exit_time,exit_price,profit,profit_pct,final
0,2026-04-23 09:16:51,191.00,2026-04-23 09:16:57,195.350,4.350,0.022775,profit
1,2026-04-23 09:17:01,195.55,2026-04-23 09:17:07,201.150,5.600,0.028637,profit
2,2026-04-23 09:21:30,200.85,2026-04-23 09:21:45,202.400,1.550,0.007717,profit
3,2026-04-23 09:28:00,198.95,2026-04-23 09:28:01,200.500,1.550,0.007791,profit
4,2026-04-23 09:28:10,205.20,2026-04-23 09:28:30,203.007,-2.193,-0.010687,loss
5,2026-04-23 09:29:00,206.20,2026-04-23 09:29:03,208.300,2.100,0.010184,profit
6,2026-04-23 09:31:00,211.00,2026-04-23 09:31:08,214.000,3.000,0.014218,profit
7,2026-04-23 09:31:10,215.55,2026-04-23 09:31:16,218.400,2.850,0.013222,profit
8,2026-04-23 09:34:20,198.95,2026-04-23 09:34:25,200.600,1.650,0.008294,profit
9,2026-04-23 09:37:10,192.40,2026-04-23 09:37:20,194.050,1.650,0.008576,profit


In [119]:
trades["final"].value_counts()

final
profit    72
loss       9
Name: count, dtype: int64

In [120]:
trades["final"].value_counts(normalize=True) * 100

final
profit    88.888889
loss      11.111111
Name: proportion, dtype: float64

In [121]:
trades[trades["final"]=="loss"].head(12)

,entry_time,entry_price,exit_time,exit_price,profit,profit_pct,final
4,2026-04-23 09:28:10,205.20,2026-04-23 09:28:30,203.007,-2.193,-0.010687,loss
31,2026-04-23 10:53:30,190.65,2026-04-23 10:55:05,188.552,-2.098,-0.011004,loss
36,2026-04-23 11:31:50,187.95,2026-04-23 11:32:59,184.877,-3.073,-0.016350,loss
38,2026-04-23 11:45:30,177.70,2026-04-23 11:47:02,175.763,-1.937,-0.010900,loss
40,2026-04-23 11:59:20,187.40,2026-04-23 12:00:13,184.681,-2.719,-0.014509,loss
55,2026-04-23 13:14:40,186.65,2026-04-23 13:15:57,184.142,-2.508,-0.013437,loss
64,2026-04-23 13:48:10,189.00,2026-04-23 13:49:04,186.102,-2.898,-0.015333,loss
78,2026-04-23 15:03:31,173.35,2026-04-23 15:04:21,170.520,-2.830,-0.016325,loss
80,2026-04-23 15:21:00,170.05,2026-04-23 15:26:34,167.580,-2.470,-0.014525,loss


In [122]:
trades[trades["final"]=="loss"].columns

Index(['entry_time', 'entry_price', 'exit_time', 'exit_price', 'profit',
       'profit_pct', 'final'],
      dtype='str')

In [123]:

# clubbed_df_2.to_excel(Path(f"assets/logs/{date_}/clubbed_df.xlsx"))

In [124]:
# clubbed_df_2["predicted"].value_counts()